# SGLT2 training and AC regularization

**Kind:** experiment.

Prepare the fixed SGLT2 population, build atom representations, then train R or U.
The selected architecture, folds, gradient scales and objective coefficients are preserved.
R enables the four AC constraints; U is the unregularized control.


In [ ]:
LOCAL_PROJECT_ROOT = "."
CONFIG_PATH = "configs/sglt2_R.json"
CHECKPOINT_PATH = "checkpoints/chemeleon_mp.pt"
DEVICE = "cpu"
RUN_MODE = "train"


In [ ]:
from pathlib import Path
import os
import sys
import json
from IPython.display import display

# Use the declared repository root in both Colab and local execution.
try:
    import google.colab
except ModuleNotFoundError as exc:
    if exc.name not in {"google", "google.colab"}:
        raise
    IS_COLAB = False
else:
    IS_COLAB = True
ROOT = Path(os.environ.get("XAI_CHEM_ROOT", LOCAL_PROJECT_ROOT)).expanduser().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print({"runtime": "colab" if IS_COLAB else "local", "root": str(ROOT), "python": sys.executable})

# Numerical tables, chemistry figures, and explicit CPU execution.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)
%matplotlib inline


## Fixed inputs and folds

In [ ]:
import gzip
import shutil

run_root = ROOT / "notebooks_for_sglt2/output/sglt2"
run_root.mkdir(parents=True, exist_ok=True)
dataset_path = ROOT / "sglt2_data/sglt2.csv"
fold_path = ROOT / "sglt2_data/sglt2_folds.csv"
mmp_path = run_root / "training_mmp.csv"
with gzip.open(ROOT / "sglt2_data/sglt2_training_mmp.csv.gz", "rb") as src, mmp_path.open("wb") as dst:
    shutil.copyfileobj(src, dst)
dataset = pd.read_csv(dataset_path)
folds = pd.read_csv(fold_path)
display(dataset[["index", "cid", "Smiles", "pIC50"]].head())
print({"molecules": len(dataset), "fold_counts": folds["fold"].value_counts().sort_index().to_dict()})


### Confirm the activity-balanced split

In [ ]:
from xai_chem.graphs import assign_activity_balanced_folds

rebuilt_folds = assign_activity_balanced_folds(dataset, n_folds=5, n_bins=5, seed=42)
# A scientific acceptance check, not a replacement split.
pd.testing.assert_frame_equal(
    rebuilt_folds[["index", "fold"]].reset_index(drop=True),
    folds[["index", "fold"]].reset_index(drop=True),
    check_dtype=False,
)
display(folds.groupby("fold")["pIC50"].agg(["count", "mean", "std"]))


## Graphs and atom mapping

In [ ]:
from xai_chem.graphs import build_graphs
from xai_chem.io import load_pyg_data
from rdkit import Chem
from rdkit.Chem import Draw

graph_root = run_root / "graphs"
RUN_GRAPH_BUILD = not graph_root.exists()
print({"run_graph_build": RUN_GRAPH_BUILD, "graph_root": str(graph_root)})
if RUN_GRAPH_BUILD:
    build_graphs(dataset_path, graph_root)

graph = load_pyg_data(graph_root / "graph_0001.pt")
display(pd.DataFrame({
    "graph_atom_index": np.arange(graph.num_nodes),
    "atomic_number": graph.atomic_numbers.numpy(),
    "smiles_to_graph_atom_index": graph.smiles_to_graph_atom_index.numpy(),
}))
display(Draw.MolToImage(Chem.MolFromSmiles(dataset.iloc[0]["Smiles"]), size=(500, 320)))
print({"graph_nodes": graph.num_nodes, "directed_edges": graph.num_edges,
       "bond_feature_shape": tuple(graph.edge_attr.shape), "node_features_before_encoder": tuple(graph.x.shape)})


## Train-only fragment frequencies

In [ ]:
from xai_chem.brics import (
    load_graph_fragment_inputs, build_brics_cache_payload,
    write_or_reuse_brics_cache, fragment_summary_frame,
)

items = load_graph_fragment_inputs(dataset_path, graph_root)
cache_root = run_root / "brics"
structure_payload = build_brics_cache_payload(items, min_heavy_atoms=1, min_count=3)
print(write_or_reuse_brics_cache(structure_payload, cache_root / "structure.pkl"))
display(fragment_summary_frame(structure_payload, len(items)).head(12))


In [ ]:
cache_audit = []
for fold in range(5):
    train_rows = set(folds.index[folds["fold"].ne(fold)])
    train_items = tuple(item for item in items if item.source_row in train_rows)
    payload = build_brics_cache_payload(
        train_items, min_heavy_atoms=1, min_count=3,
        activity_values=[item.pIC50 for item in train_items],
    )
    status = write_or_reuse_brics_cache(payload, cache_root / f"fold_{fold}_train.pkl")
    cache_audit.append({"validation_fold": fold, "training_molecules": len(train_items),
                        "fragments": len(payload["global_frag_freqs"]),
                        "positive_margin": payload["positive_margin"], "status": status})
display(pd.DataFrame(cache_audit))

output = run_root / "csv"
output.mkdir(parents=True, exist_ok=True)
pd.DataFrame(cache_audit).to_csv(output / "brics_preparation.csv", index=False)


In [ ]:
mmp = pd.read_csv(mmp_path)
print({"selected_pairs": len(mmp), "molecules": len(items),
       "atoms": sum(item.num_nodes for item in items)})
display(mmp[["pair_id", "document_id", "i_id", "j_id", "mmp_class", "dy"]].head())
display(mmp["mmp_class"].value_counts())
figure, axis = plt.subplots(figsize=(6, 3.5))
axis.hist(dataset["pIC50"], bins=25)
axis.set(xlabel="Experimental pIC50", ylabel="Molecules", title=f"Curated SGLT2 inputs (n={len(dataset)})")
display(figure)
plt.close(figure)


## Singleton CheMeleon representations

Use the official encoder checkpoint. Each molecule is encoded separately.

In [ ]:
from xai_chem.representations import load_molecule_inputs_with_provenance

loaded = load_molecule_inputs_with_provenance(
    dataset_path,
    graph_root,
    dataset_id="sglt2_inhibitors_processed_chembl3884_260729",
)
print({"molecules": len(loaded.molecules), "excluded": len(loaded.exclusions)})
display(pd.DataFrame([{"graph_idx": molecule.graph_idx, "smiles": molecule.smiles,
                       "atom_count": len(molecule.atomic_numbers)}
                      for molecule in loaded.molecules[:3]]))

In [ ]:
from xai_chem.representations import CheMeleonSingletonProvider
from xai_chem.chemistry import validate_representation_mapping

provider = CheMeleonSingletonProvider(checkpoint_path=ROOT / CHECKPOINT_PATH, device=DEVICE)
sample = loaded.molecules[0]
representation = provider.extract_one(sample)
validate_representation_mapping(sample, representation)
print({"raw_shape": representation.values.shape, "singleton": True, "dtype": str(representation.values.dtype)})
display(pd.DataFrame({"graph_atom_index": representation.graph_atom_index,
                       "atomic_number": representation.atomic_numbers,
                       "raw_feature_0": representation.values[:, 0]}).head(20))

In [ ]:
from xai_chem.representations import build_or_reuse_cache

bundle = build_or_reuse_cache(
    run_root / "representations", provider, loaded.molecules,
    input_provenance=loaded.input_provenance, provider_scoped_root=True,
)
print({"cache": str(bundle.root), "molecules": len(bundle.molecule_rows),
       "atoms": len(bundle.atomic_numbers), "dimension": bundle.values.shape[1]})
display(pd.DataFrame(bundle.molecule_rows).drop(columns="provider_metadata").head())

## Model and objective

The JSON contains the selected paper settings. Fold-specific gradient scales are fixed,
not recalibrated. Label scaling and fragment frequencies use training rows only.

$$
L=L_{\mathrm{pred}}+\lambda_m L_{\mathrm{MMP}}+\lambda_f L_{\mathrm{frag}}
$$

$$
L_{\mathrm{MMP}}=\frac{w_c s_c L_{\mathrm{core}}+w_d s_d L_{\mathrm{diff}}}{w_c+w_d},\qquad L_{\mathrm{frag}}=w_p s_p L_{\mathrm{pos}}+w_v s_v L_{\mathrm{cons}}
$$

The outer weights are `mmp_weight` and `fragment_frequency_weight`; component weights and fixed gradient scales are displayed below.


In [ ]:
from xai_chem.dataset import DataConfig, CacheBackedGraphDataset
from xai_chem.config import TrainingConfig, LossGradientScaling, CHECKED_CHEMBL3884_260729_FOLD_CONTRACTS, MMP_CORE_MAPPING_METHOD
from xai_chem.training import run_fold_training
from xai_chem.results import load_completed_fold_results, check_additivity, assert_completed_result_matches_config
from xai_chem.transfer import measure_ac_constraints

settings = json.loads((ROOT / CONFIG_PATH).read_text())
condition = Path(CONFIG_PATH).stem.removeprefix("sglt2_")
fold_numbers = settings.pop("folds")
gradient_settings = settings.pop("loss_gradient_scaling")
data_config = DataConfig(dataset_csv=dataset_path, graph_root=graph_root, fold_csv=fold_path,
                         representation_cache=run_root / "representations",
                         graph_file_template="graph_{molecule_id:04d}.pt")
raw_dataset = CacheBackedGraphDataset(data_config, validate_all=True)
configs = {}
for fold in fold_numbers:
    expected = CHECKED_CHEMBL3884_260729_FOLD_CONTRACTS[fold]
    record = gradient_settings["by_fold"].get(str(fold), gradient_settings["default"])
    configs[fold] = TrainingConfig(
        data=data_config, mmp_csv=mmp_path, fragment_cache=cache_root / f"fold_{fold}_train.pkl",
        output_dir=run_root / condition / f"fold_{fold}", device=DEVICE, validation_fold=fold,
        run_profile="strict_lightweight", mmp_pair_batch_size=128, scaling_mode="standard",
        mmp_huber_beta=0.25, fragment_min_heavy_atoms=1, fragment_min_count=3,
        fragment_min_frequency=0.1, heads=4, use_edge_features=True, atom_bias=True,
        loss_gradient_scaling=LossGradientScaling(**record),
        expected_explicit_hydrogen_exclusions=0, expected_mmp_pairs_excluded_by_hydrogen=0,
        expected_input_exclusions=0, expected_mmp_pairs_excluded_by_input=0,
        expected_mmp_source_pairs=23029, expected_mmp_symmetry_ambiguous_pairs=4845,
        expected_mmp_max_automorphism_candidates=16, expected_mmp_core_mapping_method=MMP_CORE_MAPPING_METHOD,
        **expected, **settings,
    )
display(pd.DataFrame([{"fold": fold, **config.as_dict()} for fold, config in configs.items()]).T)


## Training and raw atom contributions

In [ ]:
def print_progress(row):
    if int(row["epoch"]) == 1 or int(row["epoch"]) % 20 == 0:
        print({key: row[key] for key in ("epoch", "optimization_total", "training_rmse_raw", "validation_rmse_raw")}, flush=True)

completed = {}
for fold, config in configs.items():
    config.output_dir.parent.mkdir(parents=True, exist_ok=True)
    if RUN_MODE == "train":
        run_fold_training(config, raw_dataset=raw_dataset, epoch_observer=print_progress)
    elif RUN_MODE != "reuse":
        raise ValueError("RUN_MODE must be train or reuse")
    completed[fold] = load_completed_fold_results(config.output_dir)
    assert_completed_result_matches_config(completed[fold], config)
    display(completed[fold].history.iloc[[0, -1]])
    display(check_additivity(completed[fold].predictions, completed[fold].atom_contributions))


## Check prediction quality and AC constraints

Raw ACs sum to the prediction after adding the separate affine baseline.
Constraint values use the same loss definitions on training and validation populations.
Their reduction does not imply improved predictive accuracy or causal atomic effects.


In [ ]:
from xai_chem.plotting import plot_train_validation_yy, plot_prediction_loss_curves, save_publication_figure
for fold, result in completed.items():
    figure, scores = plot_train_validation_yy(result, run_name=condition, limits=(3.8, 9.7))
    save_publication_figure(figure, run_root / "vis" / f"{condition}_fold_{fold}_yy.png")
    display(figure); plt.close(figure)
    display(scores)
    figure = plot_prediction_loss_curves(result)
    save_publication_figure(figure, run_root / "vis" / f"{condition}_fold_{fold}_learning.png")
    display(figure); plt.close(figure)
    display(pd.DataFrame([measure_ac_constraints(result, raw_dataset, mmp_path, split=split)
                          for split in ["train", "validation"]]))
    display(result.atom_contributions.head())


In [ ]:
from xai_chem.plotting import plot_atom_contribution_grid
# One raw scale covers every displayed molecule across the configured folds.
selected_by_fold = {fold: result.predictions.query("split == 'validation'").head(2)
                    for fold, result in completed.items()}
map_limit = max(float(result.atom_contributions.loc[
    result.atom_contributions.source_row_index.isin(selected_by_fold[fold].source_row_index),
    "atom_contribution_raw"].abs().max()) for fold, result in completed.items())
print({"raw_ac_limit": map_limit, "scale_scope": "displayed molecules across configured folds",
       "condition": condition})
for fold, result in completed.items():
    selected = selected_by_fold[fold]
    figure, molecule_audit, atom_audit = plot_atom_contribution_grid(
        result, source_rows=selected.source_row_index.tolist(),
        symmetric_limit=map_limit, draw_atom_indices=True,
        contribution_label="Raw AC (pIC50); separate affine baseline",
    )
    figure_path = run_root / "vis" / f"{condition}_fold_{fold}_raw_ac.png"
    save_publication_figure(figure, figure_path)
    molecule_audit.to_csv(figure_path.with_suffix(".molecules.csv"), index=False)
    atom_audit.to_csv(figure_path.with_suffix(".atoms.csv"), index=False)
    display(figure); plt.close(figure)
    display(molecule_audit)
    display(atom_audit)


One symmetric raw AC scale covers the displayed validation molecules across all configured folds.
Its numeric limit is printed and saved in the molecule audits. Separate R/U runs can have different
limits, so compare their numerical AC tables rather than colour intensity unless the same limit
is explicitly used. Raw values are neither centred nor standardized. The affine baseline is
separate; neither the map nor in-sample Ridge fidelity establishes causal atomic effects.
